## Nädala 7: UrbanStyle kliendibaasi RFM-segmenteerimine

**Meeskond:** Tooteanalüüsi osakond  
**Kuupäev:** 13.05.2026

---

**Marko väljakutse**

UrbanStyle.ltd tootehaldur Marko Saar soovib saada paremat ülevaadet ettevõtte klientidest ja nende ostukäitumisest. Seni on ettevõte teinud otsuseid peamiselt üldiste dashboardide ja kogemuse põhjal, kuid nüüd on eesmärk liikuda andmepõhise ja personaalse turunduse suunas.

Marko soovib teada:

- Kes on ettevõtte kõige väärtuslikumad VIP-kliendid?
- Millised kliendid on ostmise lõpetamise riskis?
- Kuidas teha erinevatele kliendigruppidele paremini sihitud pakkumisi?
- Millised kliendid vajavad lojaalsusprogrammi või win-back kampaaniat?

---

**Analüüsi eesmärk**

Selle projekti eesmärk on kasutada Pythonit ja pandas teeki, et arvutada iga kliendi kohta kolm peamist RFM mõõdikut.

| Mõõdik | Kirjeldus |
|---|---|
| **Recency (R)** | Kui hiljuti klient ostis? |
| **Frequency (F)** | Kui tihti klient ostab? |
| **Monetary (M)** | Kui palju klient kulutab? |

RFM analüüsi abil saab kliendid jagada erinevatesse segmentidesse ning teha andmepõhiseid turundusotsuseid.

---

#### RFM raamistik

RFM on klassikaline kliendisegmenteerimise meetod, mida kasutavad e-kaubandusettevõtted üle maailma.

##### RFM dimensioonid

| Dimensioon | Küsimus | Arvutus |
|---|---|---|
| **Recency** | Kui hiljuti klient ostis? | Päevi viimasest ostust |
| **Frequency** | Kui tihti klient ostab? | Ostude arv perioodi jooksul |
| **Monetary** | Kui palju klient kulutab? | Kogukulutus perioodi jooksul |

##### Skoorid

- **Recency:** R=5 → ostis hiljuti, R=1 → pole ammu ostnud
- **Frequency:** F=5 → ostab tihti, F=1 → ostis ainult korra
- **Monetary:** M=5 → kulutab palju, M=1 → kulutab vähe

**RFM koguskoor = R + F + M**

In [38]:
import pandas as pd
import plotly.express as px

### Roll A: Data Loading — andmete laadimine ja liitmine

In [39]:
# 1. SAMM: Impordi pandas (juhendi punkt 1)
# (Märkus: supabase on vajalik vaid andmebaasist laadimisel)

# 2. JA 3. SAMM: Laadi sales tabel ja prindi shape ning head()
# Kasutame CSV faili vastavalt kasutaja eelistusele

df_sales = pd.read_csv('sales.csv')
print("Kuju (shape):", df_sales.shape)  # Näitab ridade ja veergude arvu [1]
print(df_sales.head())                 # Näitab esimesi ridu [1]
print()

# 4. SAMM: Laadi customers tabel ja prindi shape ning head()
df_customers = pd.read_csv('customers.csv')
print("--- SAMM 4: CUSTOMERS TABEL ---")
print("Kuju (shape):", df_customers.shape)
print(df_customers.head())
print()

# 5. SAMM: Liida tabelid customer_id põhjal (how='left') [1]
df_merged = pd.merge(
    df_sales,
    df_customers,
    on='customer_id',
    how='left'
)

# 6. SAMM: Prindi liidatud DataFrame'i info ja kontrolli veerge [1]
print("--- SAMM 6: LIIDETUD DATAFRAME ---")
print("Kuju (shape):", df.shape)
print("\nVeergude andmetüübid (dtypes):")
print(df.dtypes)

print("\nLiidetud tabeli algus (head):")
print(df.head())

# KVALITEEDIKONTROLL: Kontrollime kriitiliste veergude olemasolu [1]
required_cols = ['customer_id', 'sale_date', 'total_price', 'email']
cols_exist = all(col in df.columns for col in required_cols)

print(f"\nKVALITEEDIKONTROLL: Nõutud veerud {required_cols} olemas: {cols_exist}")

Kuju (shape): (10118, 12)
   sale_id        invoice_id   sale_date  customer_id  product_id  quantity  \
0        1  INV-202301-00001  2023-01-10       2588.0        1274         2   
1        2  INV-202301-00002  2023-01-16       4338.0        1207         2   
2        3  INV-202301-00003  2023-01-05       4673.0        1264         1   
3        4  INV-202301-00004  2023-01-02       4677.0        1341         3   
4        5  INV-202301-00005  2023-01-13       2390.0        1284         1   

   unit_price  total_price channel store_location payment_method  id  
0      234.79       469.58    pood        Tallinn          kaart   1  
1      241.13       482.26    pood          Pärnu      järelmaks   2  
2      258.46       221.19    pood          Pärnu      järelmaks   3  
3       45.21       135.63    pood          Tartu       sularaha   4  
4       99.57        99.57    pood          Tartu          kaart   5  

--- SAMM 4: CUSTOMERS TABEL ---
Kuju (shape): (3150, 9)
   customer_id f

### ROLL B: Data Cleaning - andmete puhastamine ja valideerimine

In [40]:
# Kuvame algset ridade arvu
print("Algne ridade arv:", df_merged.shape)

# Duplikaatide kontroll ja eemaldamine
print("Duplikaatide arv:", df_merged.duplicated().sum())
df_cleaned = df_merged.drop_duplicates()

# NULL väärtuste kontroll ja kriitiliste ridade eemaldamine
print("NULL väärtused veergudes:\n", df_cleaned.isnull().sum())

# Eemaldame read, kus puudub ID, kuupäev või hind
df_cleaned = df_cleaned.dropna(subset=['customer_id', 'sale_date', 'total_price'])

# Kuupäevade muutmine õigesse formaati
df_cleaned['sale_date'] = pd.to_datetime(df_cleaned['sale_date'])

# Eemaldame müügid, mis on hilisemad kui viitekuupäev
df_cleaned = df_cleaned[df_cleaned['sale_date'] <= pd.to_datetime('2025-02-28')]

# Vigaste väärtuste eemaldamine
# Jätame alles ainult need read, kus hind on suurem kui 0
df_cleaned = df_cleaned[df_cleaned['total_price'] > 0]

# Kuvame puhastusraporti, et veenduda andmestiku korrektsuses
print("\n--- PUHASTUSRAPORT ---")
print("Lõplik ridade arv (shape):", df_cleaned.shape)
print("Unikaalseid kliente:", df_cleaned['customer_id'].nunique())
print("Periood:", df_cleaned['sale_date'].min(), "kuni", df_cleaned['sale_date'].max())

# Kuvame puhastatud andmete alguse
print("\nValmis andmed analüüsiks:")
print(df_cleaned.head())

Algne ridade arv: (10118, 20)
Duplikaatide arv: 0
NULL väärtused veergudes:
 sale_id                 0
invoice_id              0
sale_date               0
customer_id           988
product_id              0
quantity                0
unit_price              0
total_price             0
channel                 0
store_location       3462
payment_method          0
id                      0
first_name            988
last_name             988
email                1944
phone                 988
city                  988
registration_date     988
loyalty_tier         4660
birth_year            988
dtype: int64

--- PUHASTUSRAPORT ---
Lõplik ridade arv (shape): (8923, 20)
Unikaalseid kliente: 2540
Periood: 2023-01-01 00:00:00 kuni 2025-02-28 00:00:00

Valmis andmed analüüsiks:
   sale_id        invoice_id  sale_date  customer_id  product_id  quantity  \
0        1  INV-202301-00001 2023-01-10       2588.0        1274         2   
1        2  INV-202301-00002 2023-01-16       4338.0        1207 

### Roll C: RFM Analysis — Recency, Frequency, Monetary + segmendid

In [41]:
# Viitekuupäev on fikseeritud, et tulemused oleksid võrreldavad
today = pd.to_datetime('2025-02-28')

# Grupeerime andmed kliendi ID järgi ja arvutame kolm olulist näitajat.
rfm = df_cleaned.groupby('customer_id').agg({
    'sale_date': lambda x: (today - x.max()).days, # Recency: päevi viimasest ostust
    'sale_id': 'count',                            # Frequency: ostude arv
    'total_price': 'sum'                           # Monetary: kogukulu
})

# Nimetame veerud ümber, et oleks selgem
rfm.columns = ['recency', 'frequency', 'monetary']

# Määra skoorid 1-5 (pd.qcut jagab andmed 5 võrdsesse gruppi)
# Recency puhul on väiksem number parem, seega sildid on tagurpidi [5, 4, 3, 2, 1]
rfm['R_score'] = pd.qcut(rfm['recency'], 5, labels=[5, 4, 3, 2, 1], duplicates='drop')
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
rfm['M_score'] = pd.qcut(rfm['monetary'], 5, labels=[1, 2, 3, 4, 5])

# Arvuta summaarne RFM skoor
rfm['RFM_Score'] = rfm['R_score'].astype(int) + rfm['F_score'].astype(int) + rfm['M_score'].astype(int)

# Segmenteerimisfunktsioon skooride põhjal
def määra_segment(score):
    if score >= 13: return 'VIP Champions'
    elif score >= 10: return 'Loyal'
    elif score >= 7: return 'Potential'
    elif score >= 4: return 'At Risk'
    else: return 'Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(määra_segment)

# VÄLJUND: RFM kokkuvõttetabel
print("--- RFM SEGMENTIDE KOKKUVÕTE ---")
segmentide_arv = rfm['Segment'].value_counts()
segmentide_protsent = rfm['Segment'].value_counts(normalize=True) * 100

kokkuvotte_df = pd.DataFrame({
    'Klientide arv': segmentide_arv,
    'Osakaal (%)': segmentide_protsent.round(1)
})

print(kokkuvotte_df)

# Kuvame tabeli alguse, et näha tulemusi
print("\nNäidis kliendiandmetest koos skooridega:")
print(rfm.head())

--- RFM SEGMENTIDE KOKKUVÕTE ---
               Klientide arv  Osakaal (%)
Segment                                  
Potential                768         30.2
Loyal                    678         26.7
At Risk                  524         20.6
VIP Champions            453         17.8
Lost                     117          4.6

Näidis kliendiandmetest koos skooridega:
             recency  frequency  monetary R_score F_score M_score  RFM_Score  \
customer_id                                                                    
2001.0            91          2    203.92       4       1       1          6   
2004.0            71          2   1198.56       4       1       4          9   
2005.0           148          4    959.60       3       4       4         11   
2006.0           477          1    327.06       1       1       1          3   
2007.0            29          1    318.63       5       1       1          7   

               Segment  
customer_id             
2001.0         At Ri

### Roll D: Visualization — 3 diagrammi + äritõlgendus

In [42]:
# Loo diagramm 1 — tulpdiagramm: segmentide klientide arv.
# Kasuta px.bar() ja lisa pealkiri eesti keeles

import plotly.express as px
import pandas as pd

# 1. Valmistame andmed ette
segment_counts = rfm['Segment'].value_counts().reset_index()
segment_counts.columns = ['Segment', 'Klientide arv']

# 2. Sorteerime suuruse järgi
segment_counts = segment_counts.sort_values(
    'Klientide arv',
    ascending=False
)

# Arvutame koguarvu ja osakaalud
total_customers = segment_counts['Klientide arv'].sum()

segment_counts['Label'] = segment_counts.apply(
    lambda x: f"{int(x['Klientide arv'])} ({x['Klientide arv']/total_customers*100:.1f}%)",
    axis=1
)

# 3. Loo tulpdiagramm
fig1 = px.bar(
    segment_counts,
    x='Segment',
    y='Klientide arv',

    text='Label',

    title=f'UrbanStyle: Kliendisegmentide jaotus (Kokku {total_customers} klienti)',

    labels={
        'Segment': 'Kliendisegment',
        'Klientide arv': 'Klientide arv'
    },

    color='Segment',

    color_discrete_map={
        'VIP Champions': '#009B8D',
        'Loyal': '#32B4A9',
        'Potential': '#78CFC6',
        'At Risk': '#FF9F43',
        'Lost': '#FF6B6B'
    }
)

# 4. Visuaalne viimistlus
fig1.update_traces(
    textposition='inside',
    textfont_size=14,
    cliponaxis=False
)

fig1.update_layout(
    plot_bgcolor='white',
    font_family='Calibri',
    showlegend=False,

    xaxis_title='Kliendisegment',
    yaxis_title='Klientide arv',

    title_font_size=22
)

fig1.show()

In [43]:
# Loo diagramm 2 — hajuvusdiagramm (scatter): recency_days vs monetary_value, värvitud segmendi järgi, suurus = frequency.
# Lisa eestikeelsed teljenimed.

# Loo diagramm 2 — hajuvusdiagramm

import plotly.express as px
import pandas as pd

# 1. ETTEVALMISTUS
rfm_plot = rfm.copy()

if 'customer_id' not in rfm_plot.columns:
    rfm_plot = rfm_plot.reset_index()

# Kui juhendi järgi on veerg monetary_value, kasutame seda
monetary_col = 'monetary_value' if 'monetary_value' in rfm_plot.columns else 'monetary'

segment_order = rfm_plot['Segment'].value_counts().index.tolist()
avg_monetary = rfm_plot[monetary_col].mean()

# Lisame kliendi nimed
customer_names = (
    df_cleaned[['customer_id', 'first_name', 'last_name']]
    .drop_duplicates()
)

rfm_plot = rfm_plot.merge(
    customer_names,
    on='customer_id',
    how='left'
)

rfm_plot['customer_name'] = (
    rfm_plot['first_name'] + ' ' + rfm_plot['last_name']
)

# 2. HAJUVUSDIAGRAMM
fig2 = px.scatter(
    rfm_plot,
    x='recency',
    y=monetary_col,
    color='Segment',
    size='frequency',
    hover_name='customer_name',
    title='UrbanStyle kliendianalüüs: VIP-kliendid toovad suurima tulu ja on aktiivseimad',
    labels={
        'recency': 'Päevi viimasest ostust',
        monetary_col: 'Kogukulutus (EUR)',
        'Segment': 'Kliendisegment',
        'frequency': 'Ostude arv'
    },
    category_orders={'Segment': segment_order},
    color_discrete_map={
        'VIP Champions': '#009B8D',
        'Loyal': '#32B4A9',
        'Potential': '#78CFC6',
        'At Risk': '#FF9F43',
        'Lost': '#FF6B6B'
    },
    hover_data={
        'customer_id': False,
        'first_name': False,
        'last_name': False
    },
    size_max=40
)

# 3. VISUAALNE VIIMISTLUS
fig2.update_layout(
    plot_bgcolor='white',
    font_family='Calibri',

    xaxis=dict(
        showgrid=False,
        showline=True,
        linecolor='#E5E7EB',
        linewidth=1,
        range=[-50, rfm_plot['recency'].max() * 1.1]
    ),

    yaxis=dict(
        showgrid=True,
        gridcolor='#F3F4F6',
        griddash='dot',
        showline=True,
        linecolor='#E5E7EB',
        linewidth=1,
        tickvals=[0, 5000, 10000, 15000, 20000, 25000, 30000],
        ticktext=['0', '5k', '10k', '15k', '20k', '25k', '30k'],
        range=[-1000, 32000]
    ),

    margin=dict(t=140, l=70, r=40, b=60)
)

# 4. KESKMISE JOON
fig2.add_hline(
    y=avg_monetary,
    line_dash="dot",
    line_color="#E5E7EB",
    annotation_text=f"Keskmine: €{avg_monetary:.0f}",
    annotation_position="top right",
    annotation_font_color="#6B7280"
)

# 5. ANNOTATSIOON
vip_top_client = rfm_plot[rfm_plot['Segment'] == 'VIP Champions'].nlargest(1, monetary_col)

target_x = vip_top_client['recency'].iloc[0]
target_y = vip_top_client[monetary_col].iloc[0]

fig2.add_annotation(
    x=target_x + 6,
    y=target_y + 2800,
    text="VIP Champions: meie väärtuslikem segment",
    showarrow=True,
    arrowhead=0,
    arrowcolor="#009B8D",
    arrowwidth=1.5,
    ax=70,
    ay=-40,
    font=dict(color="#009B8D", size=13),
    bgcolor="rgba(255, 255, 255, 0.7)",
    bordercolor="rgba(0,0,0,0)",
    xanchor='center',
    yanchor='bottom'
)

fig2.show()

In [44]:
# Loo diagramm 3 — top 10 VIP klienti kogukulutuse järgi.
# Kasuta rfm[rfm['Segment'] == 'VIP Champions'].nlargest(10, 'monetary_value')

# Diagramm 3 — TOP 10 VIP klienti kogukulutuse järgi

import plotly.express as px
import pandas as pd

# Muudame veeru nime juhendiga vastavaks
rfm = rfm.rename(columns={'monetary': 'monetary_value'})

# Võtame TOP 10 VIP klienti
top_vip = (
    rfm[rfm['Segment'] == 'VIP Champions']
    .nlargest(10, 'monetary_value')
    .reset_index()
)

# Lisame kliendi nimed
customer_names = (
    df_cleaned[['customer_id', 'first_name', 'last_name']]
    .drop_duplicates()
)

top_vip = top_vip.merge(
    customer_names,
    on='customer_id',
    how='left'
)

top_vip['customer_name'] = (
    top_vip['first_name'] + ' ' + top_vip['last_name']
)

# Loome diagrammi
fig3 = px.bar(
    top_vip,
    x='monetary_value',
    y='customer_name',
    orientation='h',

    title='UrbanStyle: TOP 10 VIP-klienti kogukulutuse järgi',

    labels={
        'monetary_value': 'Kogukulutus (EUR)',
        'customer_name': ''
    },

    text='monetary_value',

    color_discrete_sequence=['#009B8D']
)

# Tulpade otsas täpne summa
fig3.update_traces(
    texttemplate='€%{x:,.0f}',
    textposition='outside',
    cliponaxis=False
)

# Visuaalne viimistlus
fig3.update_layout(
    plot_bgcolor='white',
    font_family='Calibri',
    showlegend=False,
    bargap=0.25,

    xaxis=dict(
        showgrid=False,
        showticklabels=False,
        title=''
    ),

    yaxis=dict(
        title='',
        autorange='reversed',
        ticklabelstandoff=15
    ),

    margin=dict(t=90, l=160, r=120, b=60)
)

fig3.show()

In [45]:
# Edasijõudnute tase — Kaalutud RFM segmenteerimine

# 1. Kaalutud RFM skoor
# Monetary saab 2x kaalu, sest kliendi kulutus on äriliselt kõige olulisem

rfm['Weighted_RFM_Score'] = (
    rfm['R_score'].astype(int) +
    rfm['F_score'].astype(int) +
    (rfm['M_score'].astype(int) * 2)
)

# 2. Täpsem segmenteerimine vastavalt juhendile

def määra_advanced_segment(score):

    if score >= 13:
        return 'VIP Champions'

    elif score >= 11:
        return 'Loyal Customers'

    elif score >= 9:
        return 'Regular Customers'

    elif score >= 7:
        return 'New Customers'

    elif score >= 5:
        return 'At Risk'

    else:
        return 'Lost'

rfm['Advanced_Segment'] = rfm['Weighted_RFM_Score'].apply(
    määra_advanced_segment
)

# 3. Segmenteerimise tabel

segment_table = pd.DataFrame({
    'RFM_Score': [
        '13-15',
        '11-12',
        '9-10',
        '7-8',
        '5-6',
        '3-4'
    ],

    'Segment': [
        'VIP Champions',
        'Loyal Customers',
        'Regular Customers',
        'New Customers',
        'At Risk',
        'Lost'
    ],

    'Tegevus': [
        'Early access, VIP sooduskoodid',
        'Lojaalsusprogramm, preemiad',
        'Cross-sell kampaaniad',
        'Onboarding, welcome-sari',
        'Win-back kampaania, personaliseeritud e-mail',
        'Viimane katse, suur soodustus'
    ]
})

print("---- SEGMENTEERIMISE TABEL (EDASIJÕUDNUTE TASE) ----")
print(segment_table)

# 4. Kokkuvõte klientide arvuga

advanced_summary = (
    rfm['Advanced_Segment']
    .value_counts()
    .reset_index()
)

advanced_summary.columns = [
    'Segment',
    'Klientide arv'
]

advanced_summary['Osakaal (%)'] = (
    advanced_summary['Klientide arv'] /
    advanced_summary['Klientide arv'].sum() * 100
).round(1)

print("\n---- EDASIJÕUDNUD RFM KOKKUVÕTE ----")
print(advanced_summary)

# 5. Ekspordi CSV failina

rfm.to_csv(
    'rfm_segments.csv',
    index=True
)

print("\nCSV fail edukalt salvestatud: rfm_segments.csv")

---- SEGMENTEERIMISE TABEL (EDASIJÕUDNUTE TASE) ----
  RFM_Score            Segment                                       Tegevus
0     13-15      VIP Champions                Early access, VIP sooduskoodid
1     11-12    Loyal Customers                   Lojaalsusprogramm, preemiad
2      9-10  Regular Customers                         Cross-sell kampaaniad
3       7-8      New Customers                      Onboarding, welcome-sari
4       5-6            At Risk  Win-back kampaania, personaliseeritud e-mail
5       3-4               Lost                 Viimane katse, suur soodustus

---- EDASIJÕUDNUD RFM KOKKUVÕTE ----
             Segment  Klientide arv  Osakaal (%)
0      VIP Champions           1177         46.3
1    Loyal Customers            361         14.2
2      New Customers            355         14.0
3  Regular Customers            313         12.3
4            At Risk            217          8.5
5               Lost            117          4.6

CSV fail edukalt salvestat

### Kliendibaasi jaotuse visualiseerimine (Kaalutud RFM)

In [46]:
# Diagramm: Detailne kliendibaasi jaotus (Kaalutud RFM)
import plotly.express as px

# 1. ETTEVALMISTUS: Määrame UrbanStyle'i stiilis värvid ja segmentide järjekorra
# VIP ja Loyal kasutavad bränditoone, teised loogilist gradatsiooni
urbanstyle_colors = {
    'VIP Champions': '#009B8D',      # UrbanStyle Teal
    'Loyal Customers': '#1A1A2E',    # UrbanStyle Navy
    'Regular Customers': '#32B4A9',  # Keskmine Teal
    'New Customers': '#A5DED7',      # Hele Teal
    'At Risk': '#FF9F43',            # Oranž (Hoiatus)
    'Lost': '#FF6B6B'                # Punakas (Oht)
}

# Järjekord vastavalt Sinu edasijõudnute taseme tabelile
segment_order = ['VIP Champions', 'Loyal Customers', 'Regular Customers', 
                 'New Customers', 'At Risk', 'Lost']

# 2. ANDMETE ETTEVALMISTUS
# Arvutame klientide koguarvu dünaamiliselt pealkirja jaoks
total_customers = rfm.index.nunique()

# Kasutame Sinu loodud 'Advanced_Segment' veergu
advanced_counts = rfm['Advanced_Segment'].value_counts().reset_index()
advanced_counts.columns = ['Segment', 'Klientide arv']

# 3. LOO VERTIKAALNE TULPDIAGRAMM
fig_advanced = px.bar(
    advanced_counts,
    x='Segment',
    y='Klientide arv',
    category_orders={'Segment': segment_order},
    color='Segment',
    color_discrete_map=urbanstyle_colors,
    title=f"UrbanStyle: Strateegiline segmenteerimine (Kaalutud RFM, kokku {total_customers} klienti)",
    text='Klientide arv' # Kuvame arvud tulpade kohal
)

# 4. KUJUNDUSE VIIMISTLUS (Tufte põhimõtted)
fig_advanced.update_traces(textposition='outside')
fig_advanced.update_layout(
    plot_bgcolor='white',
    showlegend=False,
    xaxis_title="",
    yaxis_title="Klientide arv",
    font_family="Calibri"
)

fig_advanced.show()

**Kokkuvõte & peamised järeldused**

- UrbanStyle’i kliendibaas sisaldab kokku **2540 klienti**.

- Esialgses RFM mudelis kuulus segmenti **VIP Champions**:
  - **453 klienti**
  - ehk **17,8% kogu kliendibaasist**.

- Kaalutud RFM mudelis (Monetary 2x kaal) suurenes VIP segment:
  - **1177 kliendini**,
  mis näitab, et suur osa klientidest omab ettevõtte jaoks kõrget rahalist väärtust.

- Hajuvusdiagramm näitas, et VIP-klientide kogukulutus jääb enamasti vahemikku:
  - **20 000 € – 30 000 €**,
  samal ajal kui suurem osa ülejäänud klientidest kulutab alla:
  - **5000 €**.

- TOP 10 VIP klientide analüüs näitas, et:
  - suurimate klientide individuaalne kogukulutus ulatub ligi **28 000 euroni**.

- Analüüs kinnitas, et:
  - väike osa klientidest genereerib suure osa ettevõtte kogukäibest.

- At Risk segment sisaldab:
  - **524 klienti**,
  kelle aktiivsus on langenud ja kes võivad muutuda täielikult kaotatud klientideks.

- Potential segment sisaldab:
  - **768 klienti**,
  mis annab ettevõttele suure võimaluse kasvatada uusi lojaalseid püsikliente.


# Esitluse kokkuvõte

## Mis oli meie peamine JÄRELDUS?

UrbanStyle’i käive sõltub tugevalt VIP-klientidest, kelle kogukulutus on kordades suurem kui ülejäänud klientidel ning kes genereerivad suure osa ettevõtte kogutulust.

---

## Mis OTSUS muutuks selle põhjal?

Ettevõte peaks suunama rohkem ressursse VIP-klientide hoidmisele ja At Risk klientide taasaktiveerimisele läbi personaalsemate kampaaniate ja lojaalsusprogrammide.

---

## Mis meid ÜLLATAS?

Kõige suurem üllatus oli see, kui palju muutis tulemusi Monetary skoori kahekordne kaal ning kui suureks kasvas VIP segment kaalutud RFM mudelis.

## Soovitused Markole

### VIP programm
Rakendada VIP-klientidele:
- varajane ligipääs uutele toodetele,
- personaalsed VIP sooduskoodid,
- tasuta tarne ja lojaalsuspreemiad.

Eesmärk on suurendada lojaalsust ja säilitada kõige väärtuslikumad kliendid.

### Win-back kampaania
Käivitada At Risk klientidele:
- personaliseeritud „Me igatseme teid” e-mailid,
- piiratud ajaga sooduspakkumised,
- soovitused varasemate ostude põhjal.

Eesmärk on taastada klientide aktiivsus enne nende kaotamist.

### Nurture programm
Suunata Potential ja New Customers kliendid lojaalseteks püsiklientideks:
- onboarding e-mailide seeria,
- lojaalsuspunktid,
- motiveerivad sõnumid nagu „Üks ost VIP-staatuseni”.

Eesmärk on kasvatada korduvoste ja kliendi pikaajalist väärtust.

## Meeskonna refleksioon

### Mis oli suurim üllatus?
Kõige suurem üllatus oli see, kui suure osa ettevõtte käibest moodustavad VIP-kliendid. Samuti oli üllatav, kui palju muutis tulemusi Monetary skoori kahekordne kaal kaalutud RFM mudelis.

### Milline on meie peamine soovitus Markole?
Kõige olulisem tegevus on VIP-klientide hoidmine ning At Risk klientide kiire taasaktiveerimine. Need kaks segmenti mõjutavad ettevõtte käivet kõige rohkem.

### Milliseid andmeid oleks veel vaja?
Analüüsi oleks aidanud täiendada:
- kliendi vanus ja demograafilised andmed,
- toodete kategooriad,
- tagastuste info,
- veebikäitumine,
- lojaalsusprogrammi kasutus,
- NPS või kliendirahulolu andmed.